# 01. Page type review

- 확인: 분류 검증, 수동 검토, 분석 문서 구성
- 입력: 비공개 검증 결과·최종 데이터셋·분석 산출물
- 공개: URL·본문이 담긴 실행 출력 제거


In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

research_root = os.environ.get("ESG_RESEARCH_ROOT")
if not research_root:
    raise RuntimeError("ESG_RESEARCH_ROOT에 비공개 ESG 연구자료 폴더를 지정하세요.")
comp_root = Path(research_root).expanduser() / "comp"
if not comp_root.is_dir():
    raise FileNotFoundError(comp_root)

def require_file(path):
    if not path.is_file():
        raise FileNotFoundError(f"필요한 입력 파일이 없습니다: {path}")
    return path

import json
from importlib.util import module_from_spec, spec_from_file_location

phase5 = comp_root / "0_crawling" / "phase5_page_type"
validation_json = require_file(phase5 / "gt364_loo_results.json")
review_xlsx = require_file(phase5 / "저신뢰플래그_수동검토_v3.xlsx")
dataset_xlsx = require_file(phase5 / "esg_final_dataset_v3.xlsx")

# 저장된 예측만 채점합니다. 임베딩·분류 API는 다시 호출하지 않습니다.
current_dir = Path.cwd().resolve()
code_candidates = [
    candidate
    for base in (current_dir, *current_dir.parents)
    for candidate in (base, base / "code", base / "ESG_comp" / "code")
    if (candidate / "crawling" / "gt364_leave_one_out_validation.py").is_file()
]
if not code_candidates:
    raise RuntimeError("ESG_comp/code 폴더 아래 또는 포트폴리오 저장소에서 노트북을 실행하세요.")
score_path = code_candidates[0] / "crawling" / "gt364_leave_one_out_validation.py"
spec = spec_from_file_location("page_type_validation", score_path)
module = module_from_spec(spec)
spec.loader.exec_module(module)


## 분류 검증

- 저장된 수동·예측 라벨로 정확도와 Cohen’s κ 계산


In [ ]:
with validation_json.open(encoding="utf-8") as file:
    predictions = json.load(file)
correct, n, accuracy, kappa = module.score_results(predictions)
display(pd.DataFrame([{"검증 문서": n, "일치": correct, "정확도": accuracy, "Cohen's κ": kappa}]))
display(pd.crosstab(
    pd.Series([row["gt_label"] for row in predictions], name="수동 라벨"),
    pd.Series([row["pred_label"] for row in predictions], name="예측 라벨"),
))


## 수동 검토·문서 구성

- 최종 데이터셋 행 수와 전처리 후 분석 문서 수 구분
- `main` 범위는 검토 반영 페이지 유형 기준
- 원문·URL은 표시하지 않음


In [ ]:
review = pd.read_excel(review_xlsx, sheet_name="저신뢰_수동검토")
final_docs = pd.read_excel(dataset_xlsx, sheet_name="esg_final_dataset")
required_review = {"page_type", "확정_page_type"}
if not required_review.issubset(review.columns):
    raise ValueError("수동검토표의 열 구성이 예상과 다릅니다.")

output_dir = comp_root / "analysis" / "output"
analysis_docs = []
for lang in ("kr", "jp"):
    docs = pd.read_csv(require_file(output_dir / f"docs_{lang}.csv"))
    if not {"page_type_reviewed", "is_main"}.issubset(docs.columns):
        raise ValueError(f"{lang} 분석 문서의 열 구성이 예상과 다릅니다.")
    docs["country"] = lang
    analysis_docs.append(docs)
analysis_docs = pd.concat(analysis_docs, ignore_index=True)

display(pd.DataFrame([{
    "수동검토 대상": len(review),
    "확정 코드 입력": int(review["확정_page_type"].notna().sum()),
    "최종 데이터셋 행": len(final_docs),
    "전처리 후 분석 문서": len(analysis_docs),
    "main 분석 문서": int(analysis_docs["is_main"].sum()),
}]))
display(pd.crosstab(
    analysis_docs.loc[analysis_docs["is_main"], "page_type_reviewed"],
    analysis_docs.loc[analysis_docs["is_main"], "country"],
))

unresolved = review[review["확정_page_type"].isna()]
display(pd.DataFrame({"검토표 행번호": unresolved.index + 2,
                      "기존 page_type": unresolved["page_type"]}).head(20))


## 확인할 점

- 오류가 몰린 페이지 유형
- 확정 코드가 비어 있는 검토표 행
- I·J 제외 전후 문서 구성
